In [ ]:
# import os
# import pickle
# import re

# FPS = 25
# MAX_SECONDS = 6  # maximum duration for a clip

# PREFERRED_MIN = 3.0
# MIN_SECONDS = 3.0

# BOUNDARY_RE = re.compile(r"[,.?!]$")

# vsr_root = "/mnt/sdb/yuran/av_hubert/datasets/multivsr/multivsr_old"
# new_vsr_root = "/mnt/sdb/yuran/av_hubert/datasets/multivsr/multivsr"

# def clean_word(word):
#     # remove punctuation only at the beginning or end
#     word = re.sub(r"^[^\w]+|[^\w]+$", "", word)

#     return word.upper()

# def split_transcription(video_id):
#     segments = os.listdir(os.path.join(vsr_root, video_id))
#     output_folder = os.path.join(new_vsr_root, video_id)
#     os.makedirs(output_folder, exist_ok=True)

#     total_new_segments = 0

#     for segment_file in segments:
#         if not segment_file.endswith(".pckl"):
#             continue

#         base_name = segment_file.replace(".pckl", "")
#         pickle_path = os.path.join(vsr_root, video_id, base_name + ".pckl")
#         txt_path = os.path.join(vsr_root, video_id, base_name + ".txt")

#         # ---------- Load pickle ----------
#         with open(pickle_path, 'rb') as f:
#             data = pickle.load(f)

#         orig_start_frame, orig_end_frame = data["frame"]
#         orig_bbox = data["bbox"]

#         # ---------- Load txt ----------
#         with open(txt_path, 'r') as f:
#             lines = [l.strip() for l in f.readlines()]

#         text_line = lines[0]
#         lang_line = lines[1]
#         word_lines = lines[2:]

#         words = []
#         for wl in word_lines:
#             if wl and "WORD, START, END, SCORE" not in wl:
#                 parts = wl.rsplit(',', 3)
#                 if len(parts) == 4:
#                     w, s, e, sc = parts
#                     words.append({
#                         "word": w.strip(),
#                         "start": float(s),
#                         "end": float(e),
#                         "score": float(sc)
#                     })

#         if not words:
#             continue

#         # ---------- Segment words ----------
#         segments_words = []
#         current = []
#         seg_start_time = words[0]["start"]

#         for i, w in enumerate(words):
#             current.append(w)
#             duration = w["end"] - seg_start_time

#             is_boundary = bool(BOUNDARY_RE.search(w["word"]))
#             is_last = (i == len(words) - 1)

#             if (
#                 (duration >= PREFERRED_MIN and is_boundary) or
#                 duration >= MAX_SECONDS or
#                 is_last
#             ):
#                 segments_words.append(current)
#                 current = []
#                 if not is_last:
#                     seg_start_time = words[i + 1]["start"]

#         # ---------- Write new segments ----------
#         for idx, seg_words in enumerate(segments_words):
#             seg_start_t = seg_words[0]["start"]
#             seg_end_t = seg_words[-1]["end"]
            
#             if seg_end_t - seg_start_t < MIN_SECONDS:
#                 continue

#             seg_start_f = orig_start_frame + int(seg_start_t * FPS)
#             seg_end_f = orig_start_frame + int(seg_end_t * FPS)

#             # clamp
#             bbox_start = int(round(seg_start_t * FPS))
#             bbox_end   = int(round(seg_end_t * FPS))

#             bbox_start = max(0, bbox_start)
#             bbox_end   = min(len(orig_bbox) - 1, bbox_end)

#             seg_bbox = orig_bbox[bbox_start : bbox_end + 1]

#             seg_start_f = orig_start_frame + bbox_start
#             seg_end_f   = orig_start_frame + bbox_end


#             new_data = {
#                 "frame": (seg_start_f, seg_end_f),
#                 "bbox": seg_bbox
#             }

#             new_name = f"{base_name}_{idx:03d}"

#             # ---- write pckl ----
#             with open(os.path.join(output_folder, new_name + ".pckl"), "wb") as f:
#                 pickle.dump(new_data, f)

#             # ---- write txt ----
#             with open(os.path.join(output_folder, new_name + ".txt"), "w") as f:
#                 text = " ".join(clean_word(w["word"]) for w in seg_words)
#                 f.write(f"Text: {text}\n")
#                 f.write(f"{lang_line}\n\n")
#                 f.write("WORD, START, END, SCORE\n")

#                 for w in seg_words:
#                     f.write(
#                         f"{w['word']}, "
#                         f"{w['start'] - seg_start_t:.3f}, "
#                         f"{w['end'] - seg_start_t:.3f}, "
#                         f"{w['score']}\n"
#                     )

#             total_new_segments += 1

#     return total_new_segments


# all_videos = os.listdir(vsr_root)
# from tqdm import tqdm
# all_new_segments=0
# for video_id in tqdm(all_videos,total=len(all_videos)):
#     new_segs=split_transcription(video_id)
#     all_new_segments += new_segs
# print(f"Total new segments created: {all_new_segments}")

  1%|          | 34/4165 [00:01<02:58, 23.13it/s]

100%|██████████| 4165/4165 [03:04<00:00, 22.62it/s]

Total new segments created: 247932


In [2]:
# do this before preprocess (no cut version)
import os
import pickle
import shutil

FPS = 25
MIN_SECONDS = 1.0
MAX_SECONDS = 20

vsr_root = "/mnt/sdb/yuran/av_hubert/datasets/multivsr/multivsr_old"
new_vsr_root = "/mnt/sdb/yuran/av_hubert/datasets/multivsr/multivsr"
def filter_clip(video_id):
    segments = os.listdir(os.path.join(vsr_root, video_id))
    output_folder = os.path.join(new_vsr_root, video_id)
    os.makedirs(output_folder, exist_ok=True)

    kept_segments = 0

    for segment_file in segments:
        if not segment_file.endswith(".pckl"):
            continue

        base_name = segment_file.replace(".pckl", "")
        pickle_path = os.path.join(vsr_root, video_id, base_name + ".pckl")
        txt_path = os.path.join(vsr_root, video_id, base_name + ".txt")

        # ---- Load pickle ----
        with open(pickle_path, 'rb') as f:
            data = pickle.load(f)

        start_frame, end_frame = data["frame"]
        duration_seconds = (end_frame - start_frame) / FPS

        # ---- Filter by duration ----
        if MIN_SECONDS <= duration_seconds <= MAX_SECONDS:

            # ---- Determine label ----
            label = "(short)" if duration_seconds < 6 else "(long)"

            # ---- Copy pickle ----
            new_pickle_name = f"{base_name}{label}.pckl"
            shutil.copy(pickle_path,
                        os.path.join(output_folder, new_pickle_name))

            # ---- Copy txt ----
            if os.path.exists(txt_path):
                new_txt_name = f"{base_name}{label}.txt"
                shutil.copy(txt_path,
                            os.path.join(output_folder, new_txt_name))

            kept_segments += 1

    return kept_segments

# -------- Run for all videos --------
from tqdm import tqdm

all_videos = os.listdir(vsr_root)
total_kept = 0

for video_id in tqdm(all_videos, total=len(all_videos)):
    total_kept += filter_clip(video_id)

print(f"Total clips kept: {total_kept}")


100%|██████████| 4165/4165 [00:09<00:00, 440.41it/s]

Total clips kept: 41098


In [ ]:
# # run through all subfolders of /mnt/sdb/yuran/av_hubert/datasets/multivsr/multivsr
# # delete all .mp4 files
# import os
# from tqdm import tqdm
# base_folder = "/mnt/sdb/yuran/av_hubert/datasets/multivsr"
# face_folder = os.path.join(base_folder, "multivsr")
# for subfolder in tqdm(os.listdir(face_folder),total=len(os.listdir(face_folder))):
#     subfolder_path = os.path.join(face_folder, subfolder)
#     if os.path.isdir(subfolder_path):
#         for f in os.listdir(subfolder_path):
#             if f.endswith(".mp4"):
#                 pass
#                 os.remove(os.path.join(subfolder_path, f))
#                 print(f"Removed {os.path.join(subfolder_path, f)}")

100%|██████████| 4165/4165 [00:00<00:00, 30624.64it/s]
